# Deep Learning 046 — A Filter Is a Neuron

A CNN filter and an ANN neuron do the **same arithmetic**: dot product, add bias,
apply activation. The only difference is scope — a neuron sees everything, a
filter sees a window and slides. That is why backpropagation carries over
unchanged, and why **1,792** parameters can replace **9,462,592**.

| Part | What we check |
|---|---|
| A | the two are numerically identical on one patch |
| B | sliding: one filter, many outputs |
| C | 1,792 against 9,462,592 |
| D | what weight sharing actually means |

In [ ]:
import numpy as np

## Part A — Same arithmetic

In [ ]:
patch = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
filt  = np.array([[0.1, -0.2, 0.3], [0.0, 0.5, -0.1], [0.2, 0.1, -0.3]])
bias  = 0.4
relu = lambda z: np.maximum(0, z)

conv_out   = relu(np.sum(patch * filt) + bias)              # as a filter
neuron_out = relu(np.dot(patch.flatten(), filt.flatten()) + bias)   # as a neuron

print("filter on the patch:", round(float(conv_out), 6))
print("neuron on the flattened patch:", round(float(neuron_out), 6))
assert np.isclose(conv_out, neuron_out)
print("\nIdentical. The filter's values ARE weights, learned the same way.")

## Part B — The difference is scope

In [ ]:
img = np.arange(25).reshape(5, 5).astype(float)
out = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        out[i, j] = np.sum(img[i:i+3, j:j+3] * filt) + bias

print("one filter, 9 outputs:\n", np.round(out, 2))
print("\nparameters used:", filt.size + 1, "- the same 10 numbers at every position")
print("a dense layer would need", img.size, "weights PER output neuron")

## Part C — The count

In [ ]:
h, w, c = 222, 222, 3
conv = 64 * (3 * 3 * c) + 64
dense = 64 * (h * w * c) + 64
print(f"64 conv filters of 3x3x3 : {conv:,}")
print(f"dense layer of 64 units  : {dense:,}")
print(f"ratio                    : {dense / conv:,.0f}x")
assert conv == 1792 and dense == 9_462_592 and round(dense / conv / 10) * 10 == 5280

## Part D — Weight sharing, stated plainly

In [ ]:
positions = (h - 2) * (w - 2)
print(f"a 3x3 filter is applied at {positions:,} positions on this image")
print(f"and it uses the same {3*3*c + 1} numbers every time")
print(f"\na dense layer asks {positions:,} different questions;")
print(f"a conv layer asks ONE question {positions:,} times.")
print("\nThat is the saving - and the assumption: a feature worth finding in one")
print("place is worth finding in another.")

## What to take away

- A filter and a neuron compute the same thing: **dot product, bias, activation**.
- **The filter's values are weights**, learned by backpropagation as in lesson 016.
- **222×222×3: 64 filters = 1,792 parameters; 64 dense units = 9,462,592** — about
  **5,280×** more.
- The saving is **weight sharing**: one question asked everywhere.

## Exercises

1. Show the equivalence again with a 5×5 filter and a 5×5 patch.
2. How many parameters for 64 filters on a 1-channel image? Why does the channel
   count matter so much less than you might expect?
3. If a feature only ever appears in the top-left corner, is weight sharing still
   a good assumption?